# GSS Data Processing

This notebook creates a subset of the General Social Survey (GSS) data for analysis.
It extracts selected variables from the 2022 survey year.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [ ]:
DATA_DIR = Path("../data")
INPUT_FILE = DATA_DIR / "GSS_stata" / "gss7224_r2.parquet"
OUTPUT_FILE = DATA_DIR / "gss_2022.csv"

YEAR = 2022

VARIABLES = [
    "year",
    "id",
    "age",
    "sex",
    "race",
    "degree",
    "satjob",
    "hlthdep",
    # "feeldown",
    # "nointerest",
    "stress",
    "feelnerv",
    "worry",
    "wrkmeangfl",
    "richwork",
    "satfin",
    "finrela",
    "lifenow",
    "wrkstat",
    "hrs1",
    "hrs2",
    "spocc10",
    "spind10",
    "occ10", # Occupation 
    "realrinc" # Respondant income in intflation-adjusted dollars
]

In [3]:
gss_full = pd.read_parquet(INPUT_FILE)
print(f"Full dataset: {gss_full.shape[0]:,} rows, {gss_full.shape[1]} columns")

Full dataset: 75,699 rows, 6904 columns


In [4]:
gss_subset = gss_full.query("year == @YEAR")[VARIABLES].copy()
print(f"Subset: {gss_subset.shape[0]:,} rows, {gss_subset.shape[1]} columns")

Subset: 3,544 rows, 23 columns


In [5]:
gss_subset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3544 entries, 68846 to 72389
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        3544 non-null   int16  
 1   id          3544 non-null   int16  
 2   age         3336 non-null   float64
 3   sex         3524 non-null   float64
 4   race        3491 non-null   float64
 5   degree      3544 non-null   float64
 6   satjob      2463 non-null   float64
 7   hlthdep     1127 non-null   float64
 8   stress      1953 non-null   float64
 9   feelnerv    1942 non-null   float64
 10  worry       1939 non-null   float64
 11  wrkmeangfl  1953 non-null   float64
 12  richwork    1452 non-null   float64
 13  satfin      3526 non-null   float64
 14  finrela     3500 non-null   float64
 15  lifenow     1780 non-null   float64
 16  wrkstat     3536 non-null   float64
 17  hrs1        1938 non-null   float64
 18  hrs2        85 non-null     float64
 19  spocc10     1360 non-null  

In [6]:
gss_subset["realrinc"]

68846    40900.0
68847        NaN
68848    18405.0
68849     2249.5
68850        NaN
          ...   
72385        NaN
72386    27607.5
72387    33742.5
72388    27607.5
72389        NaN
Name: realrinc, Length: 3544, dtype: float64

In [7]:
missing = gss_subset.isna().sum()
missing_pct = (missing / len(gss_subset) * 100).round(1)
missing_df = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
print("Missing values per variable:")
missing_df.sort_values(by="% Missing", ascending=False)

Missing values per variable:


,Missing,% Missing
hrs2,3459,97.6
hlthdep,2417,68.2
spind10,2187,61.7
spocc10,2184,61.6
richwork,2092,59.0
lifenow,1764,49.8
worry,1605,45.3
hrs1,1606,45.3
feelnerv,1602,45.2
wrkmeangfl,1591,44.9


## Exploring FEELNERV and WORRY

These two variables measure related anxiety constructs and may be candidates for combining.

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("FEELNERV", "WORRY"))

feelnerv_counts = gss_subset["feelnerv"].value_counts().sort_index()
worry_counts = gss_subset["worry"].value_counts().sort_index()

fig.add_trace(
    go.Bar(x=feelnerv_counts.index, y=feelnerv_counts.values, name="FEELNERV"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=worry_counts.index, y=worry_counts.values, name="WORRY"),
    row=1, col=2
)

fig.update_layout(
    title_text="Distribution of FEELNERV and WORRY",
    showlegend=False,
    height=400
)
fig.update_xaxes(title_text="Response", row=1, col=1)
fig.update_xaxes(title_text="Response", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.show()

In [9]:
crosstab = pd.crosstab(gss_subset["feelnerv"], gss_subset["worry"])

fig = px.imshow(
    crosstab,
    labels=dict(x="WORRY", y="FEELNERV", color="Count"),
    title="Joint Distribution of FEELNERV and WORRY",
    text_auto=True,
    color_continuous_scale="Blues"
)
fig.update_layout(height=500, width=600)
fig.show()

In [10]:
from scipy.stats import spearmanr, kendalltau

complete_cases = gss_subset[["feelnerv", "worry"]].dropna()
n = len(complete_cases)

pearson_corr = complete_cases["feelnerv"].corr(complete_cases["worry"])
spearman_corr, spearman_p = spearmanr(complete_cases["feelnerv"], complete_cases["worry"])
kendall_corr, kendall_p = kendalltau(complete_cases["feelnerv"], complete_cases["worry"])

print(f"Correlation between FEELNERV and WORRY (n={n:,}):")
print(f"  Pearson r:  {pearson_corr:.3f}")
print(f"  Spearman ρ: {spearman_corr:.3f} (p={spearman_p:.2e})")
print(f"  Kendall τ:  {kendall_corr:.3f} (p={kendall_p:.2e})")

Correlation between FEELNERV and WORRY (n=1,933):
  Pearson r:  0.707
  Spearman ρ: 0.692 (p=4.65e-275)
  Kendall τ:  0.656 (p=2.15e-223)


## Create Anxiety Composite

Given the high correlation between FEELNERV and WORRY, we combine them into a single anxiety score using the mean.

In [11]:
# Create anxiety as mean of feelnerv and worry (uses available value if one is missing)
gss_subset["anxiety"] = gss_subset[["feelnerv", "worry"]].mean(axis=1)

print(f"Anxiety variable created:")
print(f"  Non-null values: {gss_subset['anxiety'].notna().sum():,}")
print(f"  Range: {gss_subset['anxiety'].min():.1f} - {gss_subset['anxiety'].max():.1f}")
print(f"  Mean: {gss_subset['anxiety'].mean():.2f}")
print(f"  Median: {gss_subset['anxiety'].median():.1f}")

Anxiety variable created:
  Non-null values: 1,948
  Range: 1.0 - 4.0
  Mean: 1.67
  Median: 1.5


In [12]:
gss_subset.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Saved to data/gss_2022.csv
